In [1]:
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Train.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_train = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_train = df_train.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Test.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_test = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_test = df_test.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Dev.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_val = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_val = df_val.drop(['id'], axis=1)

In [3]:
import re
import torch

# Define the mappings
aspect_to_index = {'ambience#general': 0, 'drinks#prices': 1, 'drinks#quality': 2, 'drinks#style&options': 3, 'food#prices': 4,
                   'food#quality': 5, 'food#style&options': 6, 'location#general': 7, 'restaurant#general': 8, 'restaurant#miscellaneous': 9,
                   'restaurant#prices': 10, 'service#general': 11}
# Adding 'None' to handle unspecified sentiments for any detected aspect
sentiment_to_index = {'positive': 1, 'negative': 0, 'neutral': 0, 'none': 0}

# Preprocess the labels
def preprocess_labels(label_str):
    labels = re.findall(r'\{(.*?)\}', label_str.lower())
    aspects = torch.zeros(len(aspect_to_index), dtype=torch.long)
    sentiments = torch.full((len(aspect_to_index),), sentiment_to_index['none'], dtype=torch.long)

    # Initialize with 'None'
    for label in labels:
        if ', ' in label:
            aspect, sentiment = label.split(', ')
            if aspect in aspect_to_index and sentiment in sentiment_to_index:  # Only process known aspects with sentiments
                idx = aspect_to_index[aspect]
                aspects[idx] = 1
                sentiments[idx] = sentiment_to_index[sentiment]

    return aspects, sentiments

df_train['aspects'], df_train['sentiments'] = zip(*df_train['label'].apply(preprocess_labels))
df_test['aspects'], df_test['sentiments'] = zip(*df_test['label'].apply(preprocess_labels))
df_val['aspects'], df_val['sentiments'] = zip(*df_val['label'].apply(preprocess_labels))

In [4]:
from bs4 import BeautifulSoup

for df in [df_train, df_test, df_val]:
  for sentence in range(0, len(df.text)):
    # xóa tag, link http
    processed_feature = BeautifulSoup(str(df.text[sentence]), "html.parser").get_text()
    #email-id
    processed_feature = re.sub('b[w-]+?@w+?.w{2,4}b', 'emailadd', processed_feature)
    #url
    processed_feature = re.sub('(http[s]?S+)|(w+.[A-Za-z]{2,4}S*)', 'urladd', processed_feature)

    # Xóa tất cả các ký tự đặc biệt
    processed_feature = re.sub(r'\W', ' ', processed_feature)

    # xóa từ có chứa chữ số
    # processed_feature = re.sub('W*dw*', '', processed_feature)

    # Chuyển đổi sang chữ thường
    processed_feature = processed_feature.lower()

    # Thay thế nhiều khoảng trắng bằng một khoảng trắng
    processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

    # processed_features.append(processed_feature)
    df.text[sentence] = processed_feature

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df.text[sentence] = processed_feature
<ipython-input-4-11490973c74e>:25: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/p

In [5]:
df_train['aspects'] = df_train['aspects'].apply(lambda x: x.numpy())
df_train['sentiments'] = df_train['sentiments'].apply(lambda x: x.numpy())

df_test['aspects'] = df_test['aspects'].apply(lambda x: x.numpy())
df_test['sentiments'] = df_test['sentiments'].apply(lambda x: x.numpy())

df_val['aspects'] = df_val['aspects'].apply(lambda x: x.numpy())
df_val['sentiments'] = df_val['sentiments'].apply(lambda x: x.numpy())

In [6]:
df_train

,text,label,aspects,sentiments
0,giá 53k size vừa,"{DRINKS#PRICES, neutral}, {DRINKS#STYLE&OPTION...","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,nhưng nói chung cũng hơi đắt,"{RESTAURANT#PRICES, negative}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,mình ăn rất hôi mùi dầu,"{FOOD#QUALITY, negative}","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,mình ăn chưa baoh thấy mùi hôi hải sản,"{FOOD#QUALITY, positive}","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]"
4,3 dĩa vs 2 lon revive mà có 190k thui,"{RESTAURANT#PRICES, positive}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]"
...,...,...,...,...
7023,cơ mà hôm đó bạn nv order có vẻ khó chịu với m...,"{SERVICE#GENERAL, negative}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7024,tổng hoá đơn phần bò sốt là 115k,"{FOOD#PRICES, neutral}","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7025,quán trang trí bắt mắt và sáng sủa,"{AMBIENCE#GENERAL, positive}","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
7026,trà đào ở đây đậm đà hơn ngon hơn mấy chỗ khác...,"{DRINKS#QUALITY, positive}, {RESTAURANT#MISCEL...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [7]:
import gensim

documents = [_text.split() for _text in df_train.text]
w2v_model = gensim.models.word2vec.Word2Vec(vector_size=128,
                                            window=7,
                                            min_count=10,
                                            workers=8)

w2v_model.build_vocab(documents)

w2v_model.train(documents, total_examples=len(documents), epochs=32)

(2700606, 3666912)

In [8]:
words = w2v_model.wv
vocab_size = len(words)
print("Vocab size", vocab_size)

Vocab size 1147


In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_train.text)

vocab_size = len(tokenizer.word_index) + 1
print("Total words", vocab_size)

Total words 4491


In [10]:
embedding_matrix = np.zeros((vocab_size, 128))
for word, i in tokenizer.word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i] = w2v_model.wv[word]
print(embedding_matrix.shape)

(4491, 128)


In [11]:
X_train = pad_sequences(tokenizer.texts_to_sequences(df_train.text.values), maxlen=128)
X_test = pad_sequences(tokenizer.texts_to_sequences(df_test.text.values), maxlen=128)
X_val = pad_sequences(tokenizer.texts_to_sequences(df_val.text.values), maxlen=128)

In [12]:
aspect_train = np.stack(df_train.aspects.values)
aspect_test = np.stack(df_test.aspects.values)
aspect_val = np.stack(df_val.aspects.values)

sentiment_train = np.stack(df_train.sentiments.values)
sentiment_test = np.stack(df_test.sentiments.values)
sentiment_val = np.stack(df_val.sentiments.values)

In [27]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import LSTM, Activation, Flatten, Dropout, Dense, DepthwiseConv1D, GlobalAveragePooling1D, Input, Embedding, BatchNormalization

import tensorflow.keras.backend as K

INPUT_SHAPE = (X_train.shape[1],)
N_ASPECT = 12
N_SENTIMENT = 12

class AspectSentimentNet(object):
  def _aspect_basenetwork(inputs, classes = N_ASPECT, finAct = 'softmax'):
    embedding = Embedding(input_dim=vocab_size,
                      output_dim=128,
                      weights=[embedding_matrix],
                      trainable=True)(inputs)
    x = LSTM(128, return_sequences=True)(embedding)
    x = Dropout(0.3)(x)

    x = LSTM(128, return_sequences=True)(x)
    x = Dropout(0.3)(x)

    x = LSTM(128)(x)

    # MLP classifier
    x = Dense(128, activation='relu')(x)
    x = Dense(classes)(x)
    x = Activation(finAct, name="aspect_output")(x)

    return x

  def _sentiment_basenetwork(inputs, classes = N_SENTIMENT, finAct = 'softmax'):
    embedding = Embedding(input_dim=vocab_size,
                          output_dim=128,
                          weights=[embedding_matrix],
                          trainable=True)(inputs)
    x = LSTM(128, return_sequences=True)(embedding)
    x = Dropout(0.3)(x)

    x = LSTM(128, return_sequences=True)(x)
    x = Dropout(0.3)(x)

    x = LSTM(128)(x)

    # MLP classifier
    x = Dense(128, activation='relu')(x)
    x = Dense(classes)(x)
    x = Activation(finAct, name="sentiment_output")(x)

    return x

  @staticmethod
  def build(inputShape=INPUT_SHAPE, numAspect = N_ASPECT, numSentiment = N_SENTIMENT):
    inputs = Input(shape=INPUT_SHAPE)
    aspectBranch=AspectSentimentNet._aspect_basenetwork(inputs=inputs,
      classes = numAspect, finAct='softmax')
    sentimentBranch=AspectSentimentNet._sentiment_basenetwork(inputs=inputs,
      classes = numSentiment, finAct='softmax')

    # Tạo một mô hình sử dụng đầu vào là chuỗi embedding, sau đó mô hình sẽ rẽ nhánh, một nhánh xác định đặc trưng của aspect và một nhánh xác định đặc trưng của sentiment
    model = Model(inputs=inputs,
                  outputs=[aspectBranch, sentimentBranch],
                  name="aspect_sentiment_net")

    return model

model = AspectSentimentNet.build(inputShape=INPUT_SHAPE, numAspect = N_ASPECT, numSentiment = N_SENTIMENT)


In [28]:
from tensorflow.keras.optimizers import AdamW

LR_RATE = 5e-5
EPOCHS = 100
opt = AdamW(learning_rate=LR_RATE, weight_decay=LR_RATE / EPOCHS)

losses = {"aspect_output": "binary_crossentropy",
          "sentiment_output": "categorical_crossentropy"}
metrics = {"aspect_output": "accuracy",
           "sentiment_output": "accuracy"}
model.compile(loss=losses, optimizer=opt, metrics=metrics)


In [29]:
history = model.fit(X_train, {"aspect_output": aspect_train, "sentiment_output": sentiment_train},
                    validation_data=(X_val, {"aspect_output": aspect_val, "sentiment_output": sentiment_val}),
                    batch_size=32,
                    steps_per_epoch=len(X_train) // 32,
                    epochs=EPOCHS, verbose=1)

Epoch 1/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 16s 36ms/step - aspect_output_accuracy: 0.2430 - aspect_output_loss: 0.5583 - loss: 2.2955 - sentiment_output_accuracy: 0.2070 - sentiment_output_loss: 1.7372 - val_aspect_output_accuracy: 0.3126 - val_aspect_output_loss: 0.2972 - val_loss: 1.6011 - val_sentiment_output_accuracy: 0.3061 - val_sentiment_output_loss: 1.3615
Epoch 2/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - aspect_output_accuracy: 0.2500 - aspect_output_loss: 0.1481 - loss: 1.1331 - sentiment_output_accuracy: 0.3500 - sentiment_output_loss: 0.4211 - val_aspect_output_accuracy: 0.3048 - val_aspect_output_loss: 0.2968 - val_loss: 1.6064 - val_sentiment_output_accuracy: 0.3035 - val_sentiment_output_loss: 1.3672
Epoch 3/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - aspect_output_accuracy: 0.3589 - aspect_output_loss: 0.2770 - loss: 1.5061 - sentiment_output_accuracy: 0.3475 - sentiment_output_loss: 1.2291 - val_aspect_output_accuracy: 0.4721 - val_aspect_output_loss: 0.2511 - v

In [30]:
sentence = "Nhà hàng có không gian đẹp, nhưng chất lượng món ăn lại không như mong đợi, phục vụ hơi chậm."
input = pad_sequences(tokenizer.texts_to_sequences([sentence]), maxlen=128)
prediction = model.predict(input)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step
[array([[2.0753944e-02, 2.3804378e-07, 1.2502823e-05, 6.7636656e-06,
        3.2140433e-05, 2.8973654e-01, 3.0656282e-03, 1.1647334e-05,
        2.5007900e-02, 6.1763472e-05, 2.5268705e-04, 6.6105825e-01]],
      dtype=float32), array([[0.15334563, 0.        , 0.09845299, 0.04158596, 0.00505821,
        0.2916568 , 0.15444462, 0.03021767, 0.05856692, 0.02189154,
        0.01460518, 0.13017452]], dtype=float32)]


In [31]:
score = model.evaluate(X_test, {"aspect_output": aspect_test, "sentiment_output": sentiment_test}, batch_size=32)

61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - aspect_output_accuracy: 0.6716 - aspect_output_loss: 0.1502 - loss: 2.3667 - sentiment_output_accuracy: 0.2215 - sentiment_output_loss: 2.2166


In [32]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluation(aspect_test, sentiment_test):
  y_pred = model.predict(X_test)

  aspect_pred_binary = (y_pred[0] > 0.5).astype(int)

  precision_aspect = precision_score(aspect_test, aspect_pred_binary, average='micro')
  recall_aspect = recall_score(aspect_test, aspect_pred_binary, average='micro')
  f1_aspect = f1_score(aspect_test, aspect_pred_binary, average='micro')

  sentiment_pred_class = (y_pred[0] > 0.5).astype(int)
  precision_sentiment = precision_score(sentiment_test,sentiment_pred_class, average='micro')
  recall_sentiment = recall_score(sentiment_test,sentiment_pred_class, average='micro')
  f1_sentiment = f1_score(sentiment_test,sentiment_pred_class, average='micro')

  precision = (precision_aspect + precision_sentiment) / 2
  recall = (recall_aspect + recall_sentiment) / 2
  f1 = (f1_aspect + f1_sentiment) / 2

  print("Aspect, Sentiment and Total respectively")
  print("Precision:", precision_aspect, precision_sentiment, precision)
  print("Recall:", recall_aspect, recall_sentiment, recall)
  print("F1:", f1_aspect, f1_sentiment, f1)

evaluation(aspect_test, sentiment_test)

61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
Aspect, Sentiment and Total respectively
Precision: 0.8281750266808965 0.49626467449306294 0.6622198505869797
Recall: 0.5903385317611259 0.6187624750499002 0.6045505034055131
F1: 0.6893182322895848 0.5507847201658277 0.6200514762277063
